In [10]:
import os

from dotenv import load_dotenv

# get api address from .env file
load_dotenv("../.env_vars")
EMPOWER_API_ADDRESS = os.getenv("EMPOWER_API_ADDRESS_PRD")

In [11]:
from OptiHPLCHandler import EmpowerHandler, EmpowerInstrumentMethod

In [12]:
# Create an instance of the EmpowerHandler class
handler = EmpowerHandler(project="WebAPI_test", address=EMPOWER_API_ADDRESS)
handler.connection.default_get_timeout = 120
handler.connection.default_post_timeout = 120

In [13]:
# Get the list of methods, select one, and get the method details
with handler:
    bsm_pda_method = handler.GetInstrumentMethod("@BSM_PDA_Template")
    qsm_pda_method = handler.GetInstrumentMethod("@QSM_PDA_Template")
    bsm_tuv_method = handler.GetInstrumentMethod("@BSM_TUV_Template")

In [40]:
from OptiHPLCHandler.applications.empower_implementation.empower_tools import (
    classify_eluents,
)
from OptiHPLCHandler.empower_module_method import BSMMethod

# Transfer Gradient

In [41]:
# consider names ofinputs


def transfer_gradient_table(
    input_method: EmpowerInstrumentMethod, output_method: EmpowerInstrumentMethod
) -> None:
    """Transfer the gradient table from one method to another.

    Args:
    input_method (EmpowerInstrumentMethod): The method to transfer the gradient from.
    output_method (EmpowerInstrumentMethod): The method to transfer the gradient to.

    Returns:
    EmpowerInstrumentMethod: The output method gradient table is modified in place.

    Description:
    ...

    """
    # Extract the gradient table from the input_method
    # Pump type of input_method is irrelevant as the gradient table is normalised
    input_gradient_table: list[dict] = input_method.gradient_table
    output_gradient_table: list[dict] = output_method.gradient_table

    # Determine eluent strength
    input_strong_composition = classify_eluents(input_gradient_table)["strong_eluents"][
        0
    ]
    input_weak_eluent = classify_eluents(input_gradient_table)["weak_eluents"][0]
    output_strong_composition = classify_eluents(output_gradient_table)[
        "strong_eluents"
    ][0]
    output_weak_eluent = classify_eluents(output_gradient_table)["weak_eluents"][0]

    # Determine unused solvent lines
    unused_output_solvent_lines = output_method.solvent_handler_method.solvent_lines
    unused_output_solvent_lines = [
        f"Composition{line}" for line in unused_output_solvent_lines
    ]
    unused_output_solvent_lines.remove(output_strong_composition)
    unused_output_solvent_lines.remove(output_weak_eluent)

    # Ensure strong eluent is composition B and weak eluent is composition A
    # Doesn't check if already in correct format
    new_gradient_table = []
    for step in input_gradient_table:
        new_step = step.copy()  # Copy to prevent overwiting unread composition
        # Weak eluent set to whatever the weak eluent is in output method initially
        new_step[output_weak_eluent] = step[input_weak_eluent]
        # Strong eluent set to whatever the strong eluent was in output method initially
        new_step[output_strong_composition] = step[input_strong_composition]
        # Set unused solvent lines to 0.0
        for line in unused_output_solvent_lines:
            new_step[line] = 0.0
        new_step["Time"] = step["Time"]
        new_step["Flow"] = step["Flow"]
        new_step["Curve"] = step["Curve"]
        new_gradient_table.append(new_step)
    input_gradient_table = new_gradient_table

    # Remove CompositionC and CompositionD if output method is BSM
    if isinstance(output_method.solvent_handler_method, BSMMethod):
        for step in output_gradient_table:
            step.pop("CompositionC", None)
            step.pop("CompositionD", None)

    # Transfer the gradient table to the output method
    output_method.gradient_table = input_gradient_table

In [29]:
# valves

In [43]:
# convert method from BSM to QSM
input_method = qsm_pda_method.copy()
input_method.method_name = "MethodToTransfer"

output_method = bsm_pda_method.copy()
output_method.method_name = "MethodToAdhereTo"

print("tempalate method gradient: \n", output_method.gradient_table)
print("target gradient table: \n", input_method.gradient_table)

transfer_gradient_table(input_method, output_method)
print("gradient table after transfer: \n", output_method.gradient_table)

tempalate method gradient: 
 [{'Time': 'Initial', 'Flow': '0.300', 'CompositionA': '50.0', 'CompositionB': '50.0', 'Curve': 'Initial'}, {'Time': '2.00', 'Flow': '0.300', 'CompositionA': '0.0', 'CompositionB': '100.0', 'Curve': '6'}, {'Time': '2.10', 'Flow': '0.300', 'CompositionA': '0.0', 'CompositionB': '100.0', 'Curve': '6'}, {'Time': '2.60', 'Flow': '0.300', 'CompositionA': '50.0', 'CompositionB': '50.0', 'Curve': '11'}, {'Time': '5.00', 'Flow': '0.300', 'CompositionA': '50.0', 'CompositionB': '50.0', 'Curve': '6'}]
target gradient table: 
 [{'Time': 'Initial', 'Flow': '0.400', 'CompositionA': '90.0', 'CompositionB': '10.0', 'CompositionC': '0.0', 'CompositionD': '0.0', 'Curve': 'Initial'}, {'Time': '10.00', 'Flow': '0.400', 'CompositionA': '70.0', 'CompositionB': '30.0', 'CompositionC': '0.0', 'CompositionD': '0.0', 'Curve': '6'}, {'Time': '10.10', 'Flow': '0.400', 'CompositionA': '10.0', 'CompositionB': '90.0', 'CompositionC': '0.0', 'CompositionD': '0.0', 'Curve': '6'}, {'Time': 

In [33]:
# take tests for the transfer_gradient_table function using simple name spaces as mocks for the EmpowerInstrumentMethod class
from types import SimpleNamespace
from typing import Union
from OptiHPLCHandler.empower_module_method import BSMMethod, QSMMethod


class EmpowerInstrumentMethodMock(SimpleNamespace):
    def __init__(self, gradient_table, method_name, solvent_handler_method):
        self.gradient_table: list[dict] = gradient_table
        self.method_name: str = method_name
        self.solvent_handler_method: Union[
            BSMMethod, QSMMethod
        ] = solvent_handler_method

    def copy(self):
        # make new instance of the class
        new_name = self.method_name + "_copy"
        return EmpowerInstrumentMethodMock(
            self.gradient_table, new_name, self.solvent_handler_method
        )

    def __str__(self):
        return f"Method: {self.method_name}, Gradient Table: {self.gradient_table}"


bsm_mock_initial = EmpowerInstrumentMethodMock(
    [
        {
            "Time": "Initial",
            "Flow": 0.3,
            "CompositionA": 90.0,
            "CompositionB": 10.0,
            "Curve": "Initial",
        },
        {
            "Time": 5.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "Curve": 6,
        },
        {
            "Time": 7.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "Curve": 6,
        },
        {
            "Time": 7.1,
            "Flow": 0.3,
            "CompositionA": 90.0,
            "CompositionB": 10.0,
            "Curve": 6,
        },
        {
            "Time": 10.0,
            "Flow": 0.3,
            "CompositionA": 90.0,
            "CompositionB": 10.0,
            "Curve": 6,
        },
    ],
    "BSMMock",
    BSMMethod,
)

bsm_mock_initial_2 = EmpowerInstrumentMethodMock(
    [
        {
            "Time": "Initial",
            "Flow": 0.3,
            "CompositionA": 70.0,
            "CompositionB": 30.0,
            "Curve": "Initial",
        },
        {
            "Time": 5.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "Curve": 6,
        },
        {
            "Time": 7.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "Curve": 6,
        },
        {
            "Time": 7.1,
            "Flow": 0.3,
            "CompositionA": 70.0,
            "CompositionB": 30.0,
            "Curve": 6,
        },
        {
            "Time": 10.0,
            "Flow": 0.3,
            "CompositionA": 70.0,
            "CompositionB": 30.0,
            "Curve": 6,
        },
    ],
    "BSMMock",
    BSMMethod,
)

qsm_mock_initial = EmpowerInstrumentMethodMock(
    [
        {
            "Time": "Initial",
            "Flow": 0.3,
            "CompositionA": 80.0,
            "CompositionB": 20.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": "Initial",
        },
        {
            "Time": 5.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
        {
            "Time": 7.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
        {
            "Time": 7.1,
            "Flow": 0.3,
            "CompositionA": 80.0,
            "CompositionB": 20.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
        {
            "Time": 10.0,
            "Flow": 0.3,
            "CompositionA": 80.0,
            "CompositionB": 20.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
    ],
    "QSMMock",
    QSMMethod,
)

qsm_mock_initial_2 = EmpowerInstrumentMethodMock(
    [
        {
            "Time": "Initial",
            "Flow": 0.3,
            "CompositionA": 70.0,
            "CompositionB": 30.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": "Initial",
        },
        {
            "Time": 5.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
        {
            "Time": 7.0,
            "Flow": 0.3,
            "CompositionA": 10.0,
            "CompositionB": 90.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
        {
            "Time": 7.1,
            "Flow": 0.3,
            "CompositionA": 70.0,
            "CompositionB": 30.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
        {
            "Time": 10.0,
            "Flow": 0.3,
            "CompositionA": 70.0,
            "CompositionB": 30.0,
            "CompositionC": 0.0,
            "CompositionD": 0.0,
            "Curve": 6,
        },
    ],
    "QSMMock",
    QSMMethod,
)

# Test 1 - BSM to QSM
print("Test 1 - BSM to QSM")
bsm_mock = bsm_mock_initial.copy()
qsm_mock = qsm_mock_initial.copy()
print("BSM Mock: \n", bsm_mock)
print("QSM Mock: \n", qsm_mock)
transfer_gradient_table(bsm_mock, qsm_mock)  # modifies qsm_mock in place
print("QSM Mock after transfer: \n", qsm_mock)

# Test 2 - QSM to BSM
print("Test 2 - QSM to BSM")
bsm_mock = bsm_mock_initial.copy()
qsm_mock = qsm_mock_initial.copy()
print("QSM Mock: \n", qsm_mock)
print("BSM Mock: \n", bsm_mock)
transfer_gradient_table(qsm_mock, bsm_mock)  # modifies bsm_mock in place
print("BSM Mock after transfer: \n", bsm_mock)

# Test 3 - BSM to BSM
print("Test 3 - BSM to BSM")
bsm_mock = bsm_mock_initial.copy()
bsm_mock_2 = bsm_mock_initial_2.copy()
print("BSM Mock: \n", bsm_mock)
print("BSM Mock 2: \n", bsm_mock_2)
transfer_gradient_table(bsm_mock, bsm_mock_2)  # modifies bsm_mock_2 in place
print("BSM Mock 2 after transfer: \n", bsm_mock_2)

# Test 4 - QSM to QSM
print("Test 4 - QSM to QSM")
qsm_mock = qsm_mock_initial.copy()
qsm_mock_2 = qsm_mock_initial_2.copy()
print("QSM Mock: \n", qsm_mock)
print("QSM Mock 2: \n", qsm_mock_2)
transfer_gradient_table(qsm_mock, qsm_mock_2)  # modifies qsm_mock_2 in place
print("QSM Mock 2 after transfer: \n", qsm_mock_2)

Test 1 - BSM to QSM
BSM Mock: 
 Method: BSMMock_copy, Gradient Table: [{'Time': 'Initial', 'Flow': 0.3, 'CompositionA': 90.0, 'CompositionB': 10.0, 'Curve': 'Initial'}, {'Time': 5.0, 'Flow': 0.3, 'CompositionA': 10.0, 'CompositionB': 90.0, 'Curve': 6}, {'Time': 7.0, 'Flow': 0.3, 'CompositionA': 10.0, 'CompositionB': 90.0, 'Curve': 6}, {'Time': 7.1, 'Flow': 0.3, 'CompositionA': 90.0, 'CompositionB': 10.0, 'Curve': 6}, {'Time': 10.0, 'Flow': 0.3, 'CompositionA': 90.0, 'CompositionB': 10.0, 'Curve': 6}]
QSM Mock: 
 Method: QSMMock_copy, Gradient Table: [{'Time': 'Initial', 'Flow': 0.3, 'CompositionA': 80.0, 'CompositionB': 20.0, 'CompositionC': 0.0, 'CompositionD': 0.0, 'Curve': 'Initial'}, {'Time': 5.0, 'Flow': 0.3, 'CompositionA': 10.0, 'CompositionB': 90.0, 'CompositionC': 0.0, 'CompositionD': 0.0, 'Curve': 6}, {'Time': 7.0, 'Flow': 0.3, 'CompositionA': 10.0, 'CompositionB': 90.0, 'CompositionC': 0.0, 'CompositionD': 0.0, 'Curve': 6}, {'Time': 7.1, 'Flow': 0.3, 'CompositionA': 80.0, 'C

# Transfer Wavelengths

In [34]:
def change_wavelengths(input_dict: dict, output_dict: dict) -> tuple[dict, dict]:
    """Change the wavelengths of the output dictionary to match the input dictionary.

    Args:
    input_dict (dict): The dictionary containing the input wavelengths.
    output_dict (dict): The dictionary containing the output wavelengths.

    Returns:
    tuple[dict, dict]: The dictionary containing the changes made and the output
    dictionary.

    Description:
    The function iterates over the lists, finishing at the end of the shorter list. It
    finds the wavelength of the input (whether the key be Wavelength1 or Wavelength) and
    sets the output wavelength to the input wavelength (keeping the key name the same).
    If the input does not contain the enabled key, and the output does, the enabled key
    is set to True. The function is designed to work with the following combinations:

    TUV to PDA, set all enabled to True
    PDA to TUV, takes the first two wavelengths and TUV doesn't have the enabled key
    PDA to PDA, takes the enabled key of the input and sets it to the output
    TUV to TUV, takes the first two wavelengths of the input and sets it to the output

    """
    change_dict = {}
    for input_value, (output_key, output_value) in zip(
        input_dict.values(), output_dict.items()
    ):
        input_wavelength = input_value.get("Wavelength1", input_value.get("Wavelength"))

        # Determine the wavelength key name of the output
        # (PDA: Wavelength1, TUV: Wavelength)
        output_key_name = (
            "Wavelength1" if "Wavelength1" in output_value else "Wavelength"
        )
        change_dict[output_key] = {}
        change_dict[output_key][output_key_name] = input_wavelength

        if "Enabled" not in input_value and "Enabled" in output_value:
            # TUV to PDA, set all enabled to True
            # No way of knowing if the input is enabled or not
            change_dict[output_key]["Enabled"] = True

        elif "Enabled" in input_value and "Enabled" in output_value:
            # PDA to PDA, takes enabled key of input and sets it to output
            change_dict[output_key]["Enabled"] = input_value["Enabled"]

    output_dict.update(change_dict)

    return change_dict, output_dict


pda = {
    "Channel1": {"Wavelength1": 210.0, "Enabled": True},
    "Channel2": {"Wavelength1": 220.0, "Enabled": True},
    "Channel3": {"Wavelength1": 230.0, "Enabled": False},
    "Channel4": {"Wavelength1": 240.0, "Enabled": False},
    "Channel5": {"Wavelength1": 250.0, "Enabled": False},
    "Channel6": {"Wavelength1": 260.0, "Enabled": False},
    "Channel7": {"Wavelength1": 270.0, "Enabled": False},
    "Channel8": {"Wavelength1": 280.0, "Enabled": False},
}
pda1 = {
    "Channel1": {"Wavelength1": 110.0, "Enabled": True},
    "Channel2": {"Wavelength1": 120.0, "Enabled": False},
    "Channel3": {"Wavelength1": 130.0, "Enabled": False},
    "Channel4": {"Wavelength1": 140.0, "Enabled": False},
    "Channel5": {"Wavelength1": 150.0, "Enabled": False},
    "Channel6": {"Wavelength1": 160.0, "Enabled": False},
    "Channel7": {"Wavelength1": 170.0, "Enabled": False},
    "Channel8": {"Wavelength1": 180.0, "Enabled": False},
}

tuv = {
    "Channel1": {"Wavelength": 310.0},
    "Channel2": {"Wavelength": 320.0},
}
tuv1 = {
    "Channel1": {"Wavelength": 410.0},
    "Channel2": {"Wavelength": 420.0},
}

print(input_dict := tuv)
print(output_dict := pda1)
change_wavelengths(tuv, pda1)

{'Channel1': {'Wavelength': 310.0}, 'Channel2': {'Wavelength': 320.0}}
{'Channel1': {'Wavelength1': 110.0, 'Enabled': True}, 'Channel2': {'Wavelength1': 120.0, 'Enabled': False}, 'Channel3': {'Wavelength1': 130.0, 'Enabled': False}, 'Channel4': {'Wavelength1': 140.0, 'Enabled': False}, 'Channel5': {'Wavelength1': 150.0, 'Enabled': False}, 'Channel6': {'Wavelength1': 160.0, 'Enabled': False}, 'Channel7': {'Wavelength1': 170.0, 'Enabled': False}, 'Channel8': {'Wavelength1': 180.0, 'Enabled': False}}


({'Channel1': {'Wavelength1': 310.0, 'Enabled': True},
  'Channel2': {'Wavelength1': 320.0, 'Enabled': True}},
 {'Channel1': {'Wavelength1': 310.0, 'Enabled': True},
  'Channel2': {'Wavelength1': 320.0, 'Enabled': True},
  'Channel3': {'Wavelength1': 130.0, 'Enabled': False},
  'Channel4': {'Wavelength1': 140.0, 'Enabled': False},
  'Channel5': {'Wavelength1': 150.0, 'Enabled': False},
  'Channel6': {'Wavelength1': 160.0, 'Enabled': False},
  'Channel7': {'Wavelength1': 170.0, 'Enabled': False},
  'Channel8': {'Wavelength1': 180.0, 'Enabled': False}})

In [26]:
import warnings
from typing import Union
from OptiHPLCHandler.empower_detector_module_method import PDAMethod, TUVMethod


def transfer_wavelengths(
    input_method: EmpowerInstrumentMethod, output_method: EmpowerInstrumentMethod
) -> None:
    """Transfer the wavelengths from one method to another.

    Args:
    input_method (EmpowerInstrumentMethod): The method to transfer the wavelengths from.
    output_method (EmpowerInstrumentMethod): The method to transfer the wavelengths to.

    Returns:
    EmpowerInstrumentMethod: The output method wavelengths are modified in place.

    Description:


    """
    # Find first detector method that is either PDAMethod or TUVMethod
    # If a method has both a PDA and TUV detector method, the first one is used
    input_detector_method: Union[PDAMethod, TUVMethod] = next(
        (
            detector
            for detector in input_method.detector_method_list
            if isinstance(detector, (PDAMethod, TUVMethod))
        ),
        None,
    )
    output_detector_method: Union[PDAMethod, TUVMethod] = next(
        (
            detector
            for detector in output_method.detector_method_list
            if detector.__class__.__name__ in ["PDAMethod", "TUVMethod"]
        ),
        None,
    )

    # if value is None, no PDA or TUV detector method found

    if input_detector_method is None:
        raise ValueError("No PDA or TUV detector method found in the input method.")
    if output_detector_method is None:
        raise ValueError("No PDA or TUV detector method found in the output method.")

    # Get the channel_dict
    input_detector_dict = input_detector_method.channel_dict
    output_detector_dict = output_detector_method.channel_dict

    # Transfer the wavelengths
    change_dict, _ = change_wavelengths(input_detector_dict, output_detector_dict)
    print("changed to", change_dict)

    # Set the output detector dictionary
    output_detector_method.channel_dict = change_dict

    # Set defaults
    output_detector_method.lamp_enabled = True  # Ensure lamp is enabled

    # Errors and warnings
    # If PDA method and no channels are enabled, enable the first channel
    if output_detector_method.__class__.__name__ == "PDAMethod" and not any(
        channel["Enable"] for channel in output_detector_method.channel_dict.values()
    ):
        warnings.warn(
            "No channels enabled in output method. Enabling Channel1.", stacklevel=1
        )
        output_detector_method.channel_dict["Channel1"]["Enabled"] = True

    return output_method


bsm_tuv_method_copy = bsm_tuv_method.copy()
detector = bsm_tuv_method_copy.detector_method_list[0]
detector.channel_dict = {
    "Channel1": {"Wavelength": 333.0},
}
qsm_pda_method_copy = qsm_pda_method.copy()
detector = qsm_pda_method_copy.detector_method_list[0]
detector.channel_dict = {
    "Channel1": {"Wavelength1": 666.0},
}
print(
    "BSM TUV Method: \n",
    bsm_tuv_method_copy.detector_method_list[0].channel_dict,
    "\n",
    bsm_tuv_method_copy.detector_method_list[0].channel_dict.keys(),
)
print(
    "QSM PDA Method: \n",
    qsm_pda_method_copy.detector_method_list[0].channel_dict,
    "\n",
    qsm_pda_method_copy.detector_method_list[0].channel_dict.keys(),
)
transfer_wavelengths(bsm_tuv_method_copy, qsm_pda_method_copy)
print(
    "QSM PDA Method after transfer: \n",
    qsm_pda_method_copy.detector_method_list[0].channel_dict,
    "\n",
    qsm_pda_method_copy.detector_method_list[0].channel_dict.keys(),
)

BSM TUV Method: 
 {'Channel1': {'Wavelength': '333.0', 'Type': 'Single', 'XML': '<ChannelA>\n    <Description />\n    <Wavelength>333.0</Wavelength>\n    <DataRate>SingleDataRate_20A</DataRate>\n    <DataMode>SingleMode_1A</DataMode>\n    <FilterType>Filter_2</FilterType>\n    <TimeConstant>0.1</TimeConstant>\n    <RatioMinimum>0.0001</RatioMinimum>\n    <AutoZeroWavelength>Az_3</AutoZeroWavelength>\n    <AutoZeroInjectStart>true</AutoZeroInjectStart>\n    <AutoZeroEventOrKey>true</AutoZeroEventOrKey>\n  </ChannelA>'}, 'Channel2': {'Wavelength': '214', 'Type': 'Single', 'XML': '<ChannelB>\r\n    <Description />\r\n    <Wavelength>214</Wavelength>\r\n    <DataRate>DualDataRate_1B</DataRate>\r\n    <DataMode>DualModeB_2C</DataMode>\r\n    <FilterType>Filter_2</FilterType>\r\n    <TimeConstant>2.0000</TimeConstant>\r\n    <RatioMinimum>0.0001</RatioMinimum>\r\n    <AutoZeroWavelength>Az_3</AutoZeroWavelength>\r\n    <AutoZeroInjectStart>true</AutoZeroInjectStart>\r\n    <AutoZeroEventOrKe

NameError: name 'change_wavelengths' is not defined

# Transfer Others

In [36]:
bsm_tuv_method_copy = bsm_tuv_method.copy()
qsm_pda_method_copy = qsm_pda_method.copy()
qsm_pda_method_copy.sample_temperature = 25
qsm_pda_method_copy.column_temperature = 69

print(bsm_tuv_method_copy.module_method_list[0].__dict__)

{'original_method': {'category': '', 'type': 'ARC', 'storageType': 'XML', 'name': 'ACQ-FTN', 'parentPath': '/AcquitySMDIMethod/', 'nativeXml': '<AcquitySMDIMethod language="English" version="2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="AcquityFTN-Method-R100.xsd" modified="true">\r\n  <RunTime>1.0</RunTime>\r\n  <Comment />\r\n  <LoadAhead>ModeSequential_0</LoadAhead>\r\n  <LoopOfflineEnabled>false</LoopOfflineEnabled>\r\n  <LoopOffline>LoopOfflineAutomatic_-1</LoopOffline>\r\n  <WashSolvent>Water</WashSolvent>\r\n  <WashSolventXml>&lt;Solvent version="1" permanent="true" context="all" enabled="true" date="5/1/2010 12:00:00 AM"&gt;&lt;Id&gt;144FE7DA-582C-436f-BEED-9B9B3A6EE0D0&lt;/Id&gt;&lt;Type&gt;Aqueous&lt;/Type&gt;&lt;NameId&gt;PermanentSolventWater&lt;/NameId&gt;&lt;Name&gt;Water&lt;/Name&gt;&lt;Comment /&gt;&lt;Concentration&gt;0&lt;/Concentration&gt;&lt;Viscosity&gt;1.000&lt;/Viscosity&gt;&lt;/Solvent&gt;</WashSolventXml>\r\n  <WashSolv

In [37]:
print(qsm_pda_method_copy.sample_temperature)
print(bsm_tuv_method_copy.sample_temperature)
qsm_pda_method_copy.sample_temperature = bsm_tuv_method_copy.sample_temperature
print(qsm_pda_method_copy.sample_temperature)

# shitdown all columns check

25.0
8.0
8.0


In [38]:
print(qsm_pda_method_copy.column_temperature)
print(bsm_tuv_method_copy.column_temperature)
qsm_pda_method_copy.column_temperature = bsm_tuv_method_copy.column_temperature
print(qsm_pda_method_copy.column_temperature)

69.0
45.0
45.0


In [39]:
# change name of the method by appending _transf to the name
from OptiHPLCHandler.utils.validate_method_name import append_truncate_method_name

method_name_new = append_truncate_method_name(
    qsm_pda_method_copy.method_name, "_transf"
)
qsm_pda_method_copy.method_name = method_name_new
print(qsm_pda_method_copy.method_name)

@QSM_PDA_Template_transf


In [11]:
unused_output_solvent_lines = ["A1", "B1"]

# make list comprehension to remove the last element for the line in the list if it is a digit else, keep it as it is
unused_output_solvent_lines = [
    f"Composition{line[:-1]}" if line[-1].isdigit() else f"Composition{line}"
    for line in unused_output_solvent_lines
]
unused_output_solvent_lines

['CompositionA', 'CompositionB']